# go-evo: watch the players

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrimozRavbar/go-evo/blob/main/notebooks/games_colab.ipynb)

19×19, komi 7.5, area scoring, the same rules as in training: pass only when no other move is possible, never fill your own eye, move cap 1083.

**Players**
* **PPO final 0–3**: the last 4 policies of the phase 5 PPO run (8 h of self-play).
* **PPO big 0–3**: phase 5b, the bigger PPO network (115k weights, 8 h); weaker than PPO final.
* **Evo phase 4 #0–3**: the final parents of phase 4 evolution (same network).
* **Evo 2d conv** and **Evo 3 low-rank**: champions from earlier phases.
* **random mover**: picks uniformly among the allowed moves.
* **you**: type moves such as `D4`.

`TEMPERATURE = 0` plays each network's top move (deterministic, as in all evaluations). Use 0.2–0.5 to see different games between the same two players.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("go-evo"):
        !git clone -q --depth 1 -b {BRANCH} https://github.com/PrimozRavbar/go-evo.git
    %cd go-evo
    !pip install -q numba
else:
    os.chdir(os.path.abspath(".."))  # repo root
    sys.path.insert(0, os.getcwd())

import numpy as np
from goevo.match import GameConfig, play_game
from goevo.player import Player, RandomPlayer, load_population
from goevo.viz import animate_game, slider_viewer, watch_live, draw_board
print("ready")

## Load the players

In [ ]:
class HumanPlayer:
    """Type moves like D4 or Q16 ('pass' only when no stone move is allowed)."""
    def __init__(self, name="you"):
        self.name = name
    def select_move(self, game, allowed, legal, max_moves, temperature=0.0, rng=None):
        n = game.board.shape[0]
        cols = "ABCDEFGHJKLMNOPQRST"[:n]
        while True:
            s = input(f"move {game.move_number + 1} ({'B' if game.to_play == 1 else 'W'}): ").strip().upper()
            if s in ("PASS", "P"):
                a = n * n
            else:
                try:
                    a = (n - int(s[1:])) * n + cols.index(s[0])
                except (ValueError, IndexError):
                    print("format: D4, Q16 or pass"); continue
            if 0 <= a <= n * n and allowed[a]:
                return a
            print("not allowed here (illegal, own eye, or pass while stone moves exist)")

PLAYERS = {}
def add(label, path, k=4):
    if not os.path.exists(path):
        print("missing", path); return
    genomes, cfg, *_ = load_population(path)
    for i, g in enumerate(genomes[:k]):
        PLAYERS[f"{label} {i}" if label.startswith("PPO") else f"{label} #{i}"] = Player(g, cfg, name=f"{label} {i}")
add("PPO final", "runs/p5_s0/final.pt")
add("PPO big", "runs/p5b_s0/final.pt")
add("PPO 5c", "runs/p5c_s0/final.pt")
add("PPO 5d", "runs/p5d_s0/final.pt")
add("Evo phase 4", "runs/p4_s0/gen4000.pt")
add("Evo 2d conv", "runs/d_sel_s0/gen1000.pt")
add("Evo 3 low-rank", "runs/p3_r32_s0/gen10000.pt")
PLAYERS["random mover"] = RandomPlayer("random mover")
PLAYERS["you"] = HumanPlayer()
print(list(PLAYERS))

## Play one game and replay it
Pick the players with the drop-downs, then run the cell. The animation has play, pause and step controls; `EVERY` shows every k-th position (19×19 games run to 400+ moves).

In [ ]:
BLACK = "PPO final 0"  # @param ["PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
WHITE = "Evo phase 4 #0"  # @param ["PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
TEMPERATURE = 0.0  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}
EVERY = 2  # @param {type:"integer"}

gc = GameConfig(board_size=19, temperature=TEMPERATURE)
rec = play_game(PLAYERS[BLACK], PLAYERS[WHITE], gc, np.random.default_rng(SEED))
print(f"{BLACK} (B) vs {WHITE} (W): {rec.result_string}, {len(rec.moves)} moves")
animate_game(rec, interval_ms=150, every=EVERY)

### Scrub through the same game move by move

In [ ]:
slider_viewer(rec)

### Final position

In [ ]:
import matplotlib.pyplot as plt
draw_board(rec.boards[-1], last_move=rec.moves[-1], title=f"final: {rec.result_string}")
plt.show()
print(rec.to_sgf()[:300], "...")  # full SGF in rec.to_sgf(), can be opened in any Go viewer

## Watch live
The board redraws after every move. Set `DELAY` in seconds. To play yourself, set one side to `you` and type moves in the input box.

In [ ]:
BLACK = "you"  # @param ["PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
WHITE = "PPO final 0"  # @param ["PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
TEMPERATURE = 0.0  # @param {type:"number"}
DELAY = 0.05  # @param {type:"number"}
live = watch_live(PLAYERS[BLACK], PLAYERS[WHITE], GameConfig(board_size=19, temperature=TEMPERATURE),
                  delay=DELAY, rng=np.random.default_rng(1))

## Small match
Plays `GAMES` games with alternating colours. Use `TEMPERATURE` > 0, otherwise the games repeat exactly.

In [ ]:
A = "PPO final 0"  # @param ["PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
B = "Evo phase 4 #0"  # @param ["PPO final 0", "PPO final 1", "PPO final 2", "PPO final 3", "PPO big 0", "PPO big 1", "Evo phase 4 #0", "Evo phase 4 #1", "Evo phase 4 #2", "Evo phase 4 #3", "Evo 2d conv #0", "Evo 2d conv #1", "Evo 3 low-rank #0", "Evo 3 low-rank #1", "random mover", "you"]
GAMES = 10  # @param {type:"integer"}
TEMPERATURE = 0.3  # @param {type:"number"}
gc = GameConfig(board_size=19, temperature=TEMPERATURE)
rng, wins = np.random.default_rng(0), 0
for k in range(GAMES):
    black, white = (A, B) if k % 2 == 0 else (B, A)
    r = play_game(PLAYERS[black], PLAYERS[white], gc, rng)
    a_won = (r.winner == 1) == (black == A)
    wins += a_won
    print(f"game {k + 1}: {black} (B) vs {white} (W)  {r.result_string:>12}  {len(r.moves)} moves  -> {A if a_won else B}")
print(f"{A} won {wins}/{GAMES}")